# 01 — Collect a Training Dataset

This notebook collects transition data from Procedural Frozen Lake and pushes it to the Hugging Face Hub so the same rollout data can be reused across training runs.

In [1]:
import torch

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID  = "mouse-example-dataset"  # Hub repo id for push_stores_to_hub
TOTAL_STEPS = 1500                       # env steps to collect (one step = one GroupEnv.step call)
NUM_ENVS    = 1000                       # parallel Procedural Frozen Lake instances (one Datastore each)

## Build Environment

`EnvConfig` describes the environment you want to collect from. `make_group_env` builds a `GroupEnv` that steps all instances in parallel. Importing `procedural_frozenlake` registers `Procedural-FrozenLake-v1` with Gymnasium; see the `mouse-gym` and `procedural-frozenlake` packages for the rollout interface and environment options.

Key parameters for this run:

- **One config per env instance** - `NUM_ENVS` Procedural Frozen Lake env instances are listed explicitly in `configs`. To add or remove envs, edit `NUM_ENVS`.
- **`reset_seed=i`** - each env instance gets its own reset stream.
- **`map_seed=i`** - each env instance gets a deterministic Procedural Frozen Lake map. Movement is deterministic by default (no glare ice), which keeps the oracle labels and learned behavior easy to inspect.
- **`episodes_per_task=0`** - each env instance runs one infinite task, so episode boundaries do not reset the policy context.
- **`min_width=4, max_width=8, min_height=4, max_height=8`** - maps can be as large as 8x8, matching the 64-observation vocab used by the training examples.
- **`max_episode_steps=50`** - episodes truncate after 50 steps if the agent has not reached a terminal state. This finite horizon is what pressures policies to make progress: a policy that loops without reaching the goal collects nothing before the episode ends.
- **`emit_q_star=True`** - runs value iteration for the current map and attaches the exact optimal Q-values to every step output as `info["q_star"]`. With no `step_penalty`, the env's `q_star_step_penalty` defaults to a tiny epsilon (`-1e-6`) inside value iteration only, so `argmax(q_star)` breaks ties toward shorter paths and the oracle is a real shortest-path expert while live rewards stay untouched.

In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=0,
        kwargs={
            "min_width": 4, "max_width": 8,
            "min_height": 4, "max_height": 8,
            "max_episode_steps": 50,
            "map_seed": i,
            "emit_q_star": True,  # expert Q-values in info["q_star"] every step
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)
print(f"Environments {min(10, len(env.names))} of {len(env.names)}:")
for n in env.names[:10]:
    print(n)

Environments 10 of 1000:
proc_frozenlake_0
proc_frozenlake_1
proc_frozenlake_2
proc_frozenlake_3
proc_frozenlake_4
proc_frozenlake_5
proc_frozenlake_6
proc_frozenlake_7
proc_frozenlake_8
proc_frozenlake_9


## Create Datastores

A `Datastore` is a sequential container for step data backed by a Hugging Face Arrow dataset. Each row is a plain Python dict — no fixed schema required. Whatever fields your environment returns, you can store them as-is.

The `name` you give a store becomes its config identifier on the Hub, so each parallel env instance can be fetched back by name.

In [3]:
stores = [Datastore(name=name) for name in env.names]  # one store per parallel env instance

## Collect transitions

Each step merges the input we sent (`{"action": ...}`) with the environment's output (`observation`, `reward`, `done`, ...) into a single dict and appends it to the matching store. The expert Q-values arrive nested as `info["q_star"]`; we flatten them into an `info_q_star` column so downstream objectives (`SpObjective`, `SvObjective`) can read them directly.

**Oracle ramp:** at step `t`, `oracle_prob = t / (TOTAL_STEPS - 1)`. Each env instance independently samples: with probability `oracle_prob` it picks `argmax(q_star)` (the optimal action); otherwise it picks a random action. This means the first steps explore freely while later steps demonstrate expert behavior.

**Initial reset frame:** the very first `env.step` triggers the environment reset; the env ignores the action and returns the initial observation. We store this frame so every datastore starts from timestep 0 and is time-aligned.

In [4]:
import numpy as np


def pick_inputs(outputs, env, oracle_prob: float) -> list[dict]:
    random_inputs = env.sample_random_input()
    inputs = []
    for i, out in enumerate(outputs):
        if torch.rand(1).item() >= oracle_prob:  # explore
            inputs.append(random_inputs[i])
        else:
            action = out["info"]["q_star"].argmax()  # best action per Q*
            inputs.append({"action": np.int64(action)})
    return inputs

outputs = None
env.metrics.clear()
for t in range(TOTAL_STEPS):
    oracle_prob = t / (TOTAL_STEPS - 1)         # 0.0 → 1.0 over the run
    if outputs is None:                          # first step triggers reset
        inputs = env.sample_random_input()
    else:
        inputs = pick_inputs(outputs, env, oracle_prob)
    outputs = env.step(inputs)
    for store, inp, out in zip(stores, inputs, outputs):
        row = {**inp, **out}
        info = row.pop("info")                   # flatten: keep q_star as its own column
        row["info_q_star"] = info["q_star"]
        store.append(row)

env.close()

for name, rewards, lengths in zip(env.names, env.metrics.episode_cum_rewards, env.metrics.episode_lengths):
    n = len(rewards)
    mean_r = torch.tensor(rewards).mean().item() if n else float("nan")
    mean_l = torch.tensor(lengths).mean().item() if n else float("nan")
    print(f"{name}: {n} episodes | mean reward {mean_r:.3f} | mean length {mean_l:.1f}")

proc_frozenlake_0: 56 episodes | mean reward 0.018 | mean length 24.9
proc_frozenlake_1: 68 episodes | mean reward 0.059 | mean length 21.0
proc_frozenlake_2: 60 episodes | mean reward 0.000 | mean length 23.5
proc_frozenlake_3: 95 episodes | mean reward 0.000 | mean length 14.4
proc_frozenlake_4: 128 episodes | mean reward 0.031 | mean length 10.5
proc_frozenlake_5: 76 episodes | mean reward 0.079 | mean length 18.3
proc_frozenlake_6: 87 episodes | mean reward 0.126 | mean length 16.2
proc_frozenlake_7: 230 episodes | mean reward 0.009 | mean length 5.4
proc_frozenlake_8: 70 episodes | mean reward 0.057 | mean length 19.9
proc_frozenlake_9: 93 episodes | mean reward 0.022 | mean length 14.8
proc_frozenlake_10: 186 episodes | mean reward 0.000 | mean length 6.9
proc_frozenlake_11: 185 episodes | mean reward 0.492 | mean length 7.1
proc_frozenlake_12: 174 episodes | mean reward 0.040 | mean length 7.6
proc_frozenlake_13: 239 episodes | mean reward 0.678 | mean length 5.3
proc_frozenlake

## Push to the Hub

Upload all stores to your Hugging Face account as a single dataset repo. Each store is saved as a separate named config, which keeps the parallel env streams distinct when the dataset is loaded later.

In [5]:
url = push_stores_to_hub(stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")

Pushed to micahr234/mouse-example-dataset (proc_frozenlake_0/train: 1500, proc_frozenlake_1/train: 1500, proc_frozenlake_2/train: 1500, proc_frozenlake_3/train: 1500, proc_frozenlake_4/train: 1500, proc_frozenlake_5/train: 1500, proc_frozenlake_6/train: 1500, proc_frozenlake_7/train: 1500, proc_frozenlake_8/train: 1500, proc_frozenlake_9/train: 1500, proc_frozenlake_10/train: 1500, proc_frozenlake_11/train: 1500, proc_frozenlake_12/train: 1500, proc_frozenlake_13/train: 1500, proc_frozenlake_14/train: 1500, proc_frozenlake_15/train: 1500, proc_frozenlake_16/train: 1500, proc_frozenlake_17/train: 1500, proc_frozenlake_18/train: 1500, proc_frozenlake_19/train: 1500, proc_frozenlake_20/train: 1500, proc_frozenlake_21/train: 1500, proc_frozenlake_22/train: 1500, proc_frozenlake_23/train: 1500, proc_frozenlake_24/train: 1500, proc_frozenlake_25/train: 1500, proc_frozenlake_26/train: 1500, proc_frozenlake_27/train: 1500, proc_frozenlake_28/train: 1500, proc_frozenlake_29/train: 1500, proc_fr